# Dunnhumby M2 단일시드 test-only 실험

이번 실행은 `seed=42`에서 M1@64와 수정 M2만 빠르게 비교합니다.

- 기존 train과 validation을 병합해 학습
- 두 모형 모두 100 epoch 고정(조기종료·epoch 선택 없음)
- test는 각 모형의 최종 체크포인트에서 한 번만 평가
- holdout은 생성·평가하지 않음
- epoch별 자동 저장·재개 지원

> 이 test 구간은 과거 실험에서 이미 노출되었으므로, 이번 결과는 최종 확증이 아니라 구조 개선을 위한 탐색적 점검으로 해석합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '16d03a8861447d6a1afdf107ad7d154cc371ff9d'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
from pathlib import Path
from IPython.display import display

from lightgcn_clv_postprop_gate_test1 import (
    configure_test1_run, preflight_summary, run_test1,
)

cfg = configure_test1_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_postprop_axis_gate_test1_v1'
    ),
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


## 저장된 진행상태 확인(선택)

아래 셀은 학습을 시작하지 않습니다. 재연결 후 저장된 진행상태를 확인할 때만 실행하면 됩니다.

In [ ]:
progress_files = sorted(Path(cfg.out_dir).glob('progress/*/progress.json'))
if not progress_files:
    print('아직 저장된 progress.json이 없습니다.')
else:
    for path in progress_files:
        print(path)
        print(json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2))


## M1과 수정 M2 실행

아래 셀 하나만 실행하면 M1과 M2를 순차 학습합니다. 연결이 끊기면 위 셀부터 다시 실행한 뒤 이 셀을 다시 누르면 저장된 epoch 다음부터 이어집니다.

In [ ]:
result_df = run_test1(cfg)


In [ ]:
import pandas as pd

comparison_display = result_df.attrs['comparison']
if not isinstance(comparison_display, pd.DataFrame):
    comparison_display = pd.DataFrame(comparison_display)
absolute_display = result_df.copy()
absolute_display.attrs = {}

print('seed 42 test 절대지표:')
display(absolute_display)
print('M1 대비 변화:')
display(comparison_display)
print('결과 파일:')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
